# Build and test a tiny causal decoder

Implement a framework-free decoder on small fixed tensors. Write down all dimensions before combining the exercises.

Fill in each starter, then run its check cell. Restart the kernel and run all cells before submitting. Keep your interpretation in the last Markdown cell.


## 1. Implement scaled query key scores

Compute scaled attention scores.

Implement scaled_scores(query, keys). Reject an empty query or any mismatched key width. Return one scaled score per key, in input order.


In [ ]:
def scaled_scores(query,keys):
    raise NotImplementedError


In [ ]:
assert scaled_scores([1,0,0,0],[[2,0,0,0],[0,2,0,0]]) == [1,0]
assert scaled_scores([2],[[3],[-1]]) == [6,-2]
assert scaled_scores([1],[]) == []
try:
    scaled_scores([1,2],[[1]])
except ValueError:
    pass
else:
    raise AssertionError("reject incompatible key width")
try:
    scaled_scores([],[])
except ValueError:
    pass
else:
    raise AssertionError("reject an empty query")
print("OK")


## 2. Implement causal attention weights

Prevent future positions from influencing a prefix.

Implement causal_weights(scores, position) for finite logits and integer position. Return probabilities with keys after position exactly zero. Reject indices outside the row.


In [ ]:
def causal_weights(scores,position):
    raise NotImplementedError


In [ ]:
assert causal_weights([1000,1000,5000],1)==[0.5,0.5,0]
assert causal_weights([0,1],0)==[1,0]
assert causal_weights([0,0],1)==[0.5,0.5]
try:
    causal_weights([],0)
except ValueError:
    pass
else:
    raise AssertionError("reject invalid query index")
try:
    causal_weights([0,1],2)
except ValueError:
    pass
else:
    raise AssertionError("reject a query index past the row")
try:
    causal_weights([0,1,2],-2)
except ValueError:
    pass
else:
    raise AssertionError("reject a negative query index")
print("OK")


## 3. Implement concatenate attention heads

Concatenate multiple heads in per-token order.

Implement concatenate_heads(heads) for a list of token-by-feature lists. Concatenate within each token, preserving head order. Reject different token counts. An empty head list yields [].


In [ ]:
def concatenate_heads(heads):
    raise NotImplementedError


In [ ]:
assert concatenate_heads([[[1],[2]],[[3,4],[5,6]]])==[[1,3,4],[2,5,6]]
assert concatenate_heads([])==[]
assert concatenate_heads([[],[]])==[]
try:
    concatenate_heads([[[1]],[[1],[2]]])
except ValueError:
    pass
else:
    raise AssertionError("reject token mismatch")
print("OK")


## 4. Implement token-wise layer normalisation

Check a pre-normalized residual block.

Implement normalize_token(row, epsilon=1e-5) with population variance within one token row and epsilon inside the square root. Reject empty rows and nonpositive epsilon.


In [ ]:
def normalize_token(row,epsilon=1e-5):
    raise NotImplementedError


In [ ]:
a=normalize_token([1,3],3)
assert a==[-0.5,0.5]
assert normalize_token([7,7])==[0,0]
assert normalize_token([2])==[0]
try:
    normalize_token([],1)
except ValueError:
    pass
else:
    raise AssertionError("reject empty row")
try:
    normalize_token([1,3],0)
except ValueError:
    pass
else:
    raise AssertionError("reject zero epsilon")
print("OK")


## 5. Compose and test a causal decoder block

Implement decoder_block(rows) for a sequence of width-four embeddings. Add positional offset
t/10 to the first feature of row t. Normalize each offset row. Split normalized rows into two
heads of width two, using the same values for queries, keys and values (identity projections).
Each head computes scaled scores, causal weights, and a weighted value sum. Concatenate the heads
per token; use identity output projection. Add the result to the offset residual. Normalize
the updated row and add the position-wise transform 0.5×ReLU of each normalized feature.
Return the final rows. An empty input returns []. This fixed-weight block illustrates data flow,
not a trained language model. Check that a modified suffix cannot change the prefix.


In [ ]:
def decoder_block(rows):
    raise NotImplementedError


In [ ]:
prefix = [[1.,0.,-1.,2.],[2.,0.,0.,-1.]]
standalone = decoder_block(prefix)
assert decoder_block(prefix+[[500.,-200.,-20.,100.]])[:2] == standalone
assert decoder_block(prefix+[[-500.,200.,20.,-100.]])[:2] == standalone
assert len(standalone)==2 and all(len(row)==4 for row in standalone)
assert decoder_block([[0,0,0,0]]) == [[0,0,0,0]]
assert decoder_block([]) == []
row = prefix[0]
attended = [x+y for x,y in zip(row,normalize_token(row))]
expected = [x+0.5*max(0,y) for x,y in zip(attended,normalize_token(attended))]
assert standalone[0] == expected and standalone[0] != row
expected1 = [4.290691696601262, -0.2870688252734473, -0.401624090875215, -1.7724062331950405]
assert all(abs(a-b) < 1e-9 for a,b in zip(standalone[1], expected1))
print("OK")


## Interpretation

Change only the last input token and compare all previous outputs. Explain which part of the implementation would leak information if the causal mask were removed.

**Your answer**
